# CyberForecaster — LSTM Training on Colab

## What to upload (all four items — training fails without any of them)

On the left sidebar click the **Folder icon** (Files), create `data/processed/`, and upload:

1. `sequences_train.npz`
2. `sequences_val.npz`
3. `sequences_test.npz`
4. **`scaler.npz`** ← easy to forget; training raises without it

Then upload your whole `src` folder so it sits next to `data` in the root.

> `scaler.npz` is the input transform fitted on the training split only.
> Training and inference must share it: a model trained on scaled features and
> then fed raw ones in the app produces confident nonsense with no error.

In [ ]:
# 1. Dependencies (Colab already ships torch + CUDA)
!pip install -q captum scikit-learn

In [ ]:
# 2. Verify the upload BEFORE burning GPU time on a missing file
import pathlib, numpy as np

need = ['sequences_train.npz', 'sequences_val.npz', 'sequences_test.npz', 'scaler.npz']
missing = [f for f in need if not pathlib.Path('data/processed', f).exists()]
assert not missing, f'MISSING UPLOADS: {missing}'
assert pathlib.Path('src/models/lstm_forecaster.py').exists(), 'upload the src/ folder'

d = np.load('data/processed/sequences_train.npz')
print('X       ', d['X'].shape)
print('y_prog  ', d['y_prog'].shape, '<- must be (n, K), NOT (n,)')
assert d['y_prog'].ndim == 2, 'y_prog is 1-D: re-run the pipeline locally and re-upload'
print('ends    ', 'present' if 'ends' in d.files else 'MISSING (re-run pipeline)')
print('features', [str(n) for n in d['feature_names']])

In [ ]:
# 3. Train
!python -m src.models.lstm_forecaster --dir data/processed --epochs 40

In [ ]:
# 4. Lead-time evaluation — the differentiator slide. Run it HERE, while the
#    trained model is still on this machine.
!python -m src.evaluation.lead_time --dir data/processed

In [ ]:
# 5. Bundle everything you must bring home.
#    Downloading only the .pt (the old instruction) loses the test metrics —
#    which is exactly why the Benchmark tab had no LSTM row.
!zip -j cyberforecaster_trained.zip \
    models/trained_models/lstm_forecaster.pt \
    models/trained_models/lstm_config.json \
    models/metrics_lstm.json \
    models/metrics_lead_time.json

from google.colab import files
files.download('cyberforecaster_trained.zip')

### Back on your laptop

```bash
unzip -o cyberforecaster_trained.zip -d /tmp/tr
cp /tmp/tr/lstm_forecaster.pt  models/trained_models/
cp /tmp/tr/lstm_config.json    models/trained_models/
cp /tmp/tr/metrics_*.json      models/

python scripts/verify_state.py        # must show feature counts agreeing
python scripts/build_demo_cache.py    # freeze the crash-proof demo
streamlit run app/streamlit_app.py
```

`verify_state.py` fails loudly on a feature-count mismatch between `scaler.npz`,
the npz files and `lstm_config.json`. Never demo a mismatch — the app would
silently mispredict.